# Spanning Trees and Shortest Paths



```{contents}
:local:
:depth: 2
```


Weighted graphs turn relationships into optimization problems. A minimum spanning tree connects all vertices as cheaply as possible. A shortest-path tree finds low-cost routes from one source vertex.


```{index} minimum spanning tree
```

## Minimum Spanning Trees

A spanning tree connects every vertex without cycles. A **minimum spanning tree** (MST) has the lowest possible total edge cost among all spanning trees.


In [ ]:
using System;
using System.Linq;

var edges = new[]
{
    new Edge("A", "B", 4),
    new Edge("A", "C", 2),
    new Edge("B", "C", 1),
    new Edge("B", "D", 5),
    new Edge("C", "D", 8)
};

foreach (Edge edge in edges.OrderBy(edge => edge.Cost))
{
    Console.WriteLine($"{edge.From}-{edge.To}: {edge.Cost}");
}

record Edge(string From, string To, int Cost);


The cheapest edge list is not yet the answer. The algorithm must avoid cycles while connecting all vertices.


```{index} Kruskal algorithm
```

## Kruskal's Algorithm

Kruskal's algorithm sorts edges by cost and accepts an edge when it connects two components that were previously separate.


In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var vertices = new[] { "A", "B", "C", "D" };
var edges = new[]
{
    new Edge("A", "B", 4),
    new Edge("A", "C", 2),
    new Edge("B", "C", 1),
    new Edge("B", "D", 5),
    new Edge("C", "D", 8)
};

var parent = vertices.ToDictionary(vertex => vertex, vertex => vertex);
var mst = new List<Edge>();

foreach (Edge edge in edges.OrderBy(edge => edge.Cost))
{
    if (Find(edge.From) != Find(edge.To))
    {
        mst.Add(edge);
        Union(edge.From, edge.To);
    }
}

Console.WriteLine($"total cost = {mst.Sum(edge => edge.Cost)}");
foreach (Edge edge in mst)
{
    Console.WriteLine($"{edge.From}-{edge.To}: {edge.Cost}");
}

string Find(string vertex)
{
    while (parent[vertex] != vertex)
    {
        vertex = parent[vertex];
    }
    return vertex;
}

void Union(string a, string b)
{
    parent[Find(a)] = Find(b);
}

record Edge(string From, string To, int Cost);


The invariant is that the accepted edges form a forest: no cycles are introduced. Each accepted edge connects two previously separate components.


```{index} shortest path; Dijkstra algorithm
```

## Shortest Paths With Nonnegative Weights

Dijkstra's algorithm repeatedly settles the unsettled vertex with the smallest known distance. It works when edge weights are nonnegative.


In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new Dictionary<string, List<(string To, int Cost)>>
{
    ["A"] = new() { ("B", 4), ("C", 2) },
    ["B"] = new() { ("D", 5) },
    ["C"] = new() { ("B", 1), ("D", 8) },
    ["D"] = new()
};

var distances = Dijkstra(graph, "A");
foreach ((string vertex, int distance) in distances.OrderBy(item => item.Key))
{
    Console.WriteLine($"A -> {vertex}: {distance}");
}

Dictionary<string, int> Dijkstra(Dictionary<string, List<(string To, int Cost)>> graph, string source)
{
    var distance = graph.Keys.ToDictionary(vertex => vertex, _ => int.MaxValue);
    var queue = new PriorityQueue<string, int>();
    distance[source] = 0;
    queue.Enqueue(source, 0);

    while (queue.Count > 0)
    {
        string current = queue.Dequeue();
        foreach ((string to, int cost) in graph[current])
        {
            int candidate = distance[current] + cost;
            if (candidate < distance[to])
            {
                distance[to] = candidate;
                queue.Enqueue(to, candidate);
            }
        }
    }

    return distance;
}


The greedy choice is the smallest known tentative distance. With nonnegative weights, a shorter path to that vertex cannot appear later through an unsettled vertex.


```{index} shortest path; path reconstruction
```

## Reconstructing Paths

Distances answer "how much?" A predecessor table answers "which route?"


In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new Dictionary<string, List<(string To, int Cost)>>
{
    ["A"] = new() { ("B", 4), ("C", 2) },
    ["B"] = new() { ("D", 5) },
    ["C"] = new() { ("B", 1), ("D", 8) },
    ["D"] = new()
};

var previous = new Dictionary<string, string?>();
var distance = graph.Keys.ToDictionary(vertex => vertex, _ => int.MaxValue);
var queue = new PriorityQueue<string, int>();

distance["A"] = 0;
previous["A"] = null;
queue.Enqueue("A", 0);

while (queue.Count > 0)
{
    string current = queue.Dequeue();
    foreach ((string to, int cost) in graph[current])
    {
        int candidate = distance[current] + cost;
        if (candidate < distance[to])
        {
            distance[to] = candidate;
            previous[to] = current;
            queue.Enqueue(to, candidate);
        }
    }
}

var path = new Stack<string>();
for (string? at = "D"; at is not null; at = previous.GetValueOrDefault(at))
{
    path.Push(at);
}

Console.WriteLine($"distance = {distance["D"]}");
Console.WriteLine(string.Join(" -> ", path));


Path reconstruction is usually a small addition to the shortest-path algorithm, but it makes results much more useful.


```{index} graph optimization; algorithm selection
```

## MST or Shortest Path?

Minimum spanning trees and shortest paths solve different problems. MSTs design a low-cost network connecting every vertex. Shortest paths find low-cost routes from a source to destinations.


In [ ]:
using System;

string goal = "connect every office with minimum cable cost";

if (goal.Contains("every") && goal.Contains("connect"))
{
    Console.WriteLine("Think minimum spanning tree.");
}
else
{
    Console.WriteLine("Think shortest path or another route problem.");
}


Choosing the right model is part of algorithm design. The same weighted graph can support different questions.


```{rubric} Footnotes
```
[^1]: Kruskal's algorithm is one of several MST algorithms. Prim's algorithm is another common greedy MST algorithm.
[^2]: Dijkstra's algorithm does not handle negative edge weights correctly. Later courses introduce algorithms such as Bellman-Ford for that setting.
